In [28]:
from pathlib import Path
import sys
import pandas as pd
import io
from typing import Callable
import json

# 노트북 기준 상위 폴더를 프로젝트 루트로 가정
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("현재 경로:", Path.cwd())

from llm.openrouter import chat

현재 경로: /Users/a114384/Desktop/멘토링/compliance_agent/v1_text


In [ ]:
def _strip_json_fence(text: str) -> str:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        lines = cleaned.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].startswith("```"):
            lines = lines[:-1]
        cleaned = "\n".join(lines).strip()
    return cleaned

def generate_text(
    dataset_name: str,
    topic: str,
    count: int,
    output_csv_path: str,
    system_prompt: str) -> str:
    user_prompt = f"""
Dataset name: {dataset_name}
Topic: {topic}
Sentence Rows: {count}

Requirements:
- 금융 도메인 기반의 마스킹 텍스트 데이터셋을 생성하세요.
- JSON 배열(Array)만 반환하세요. 마크다운 코드펜스 금지.
- 각 원소는 raw_text, masked_text, masked_word 키를 모두 가져야 합니다.
- 모든 값은 문자열로 생성하세요.
""".strip()

    system_prompt = system_prompt.strip()
    raw_response = chat(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
    )
    cleaned = _strip_json_fence(raw_response)
    data = json.loads(cleaned)

    if not isinstance(data, list):
        raise ValueError("LLM 응답은 JSON 배열(list)이어야 합니다.")

    expected_keys = {"raw_text", "masked_text", "masked_word"}
    normalized_rows = []
    for i, row in enumerate(data[:count]):
        if not isinstance(row, dict):
            raise ValueError(f"{i}번째 항목이 dict가 아닙니다.")

        missing = expected_keys - set(row.keys())
        if missing:
            raise ValueError(f"{i}번째 항목 키 누락: {missing}")

        normalized_rows.append(
            {
                "raw_text": str(row["raw_text"]),
                "masked_text": str(row["masked_text"]),
                "masked_word": str(row["masked_word"]),
            }
        )

    df = pd.DataFrame(normalized_rows, columns=["raw_text", "masked_text", "masked_word"])

    output_path = Path(output_csv_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False, encoding="utf-8-sig")

    return str(output_path)

In [ ]:
system_prompt = """
고품질 금융 마스킹 데이터셋을 생성하세요.
반드시 JSON 배열(Array)만 반환하세요. 마크다운 코드펜스 금지.
각 원소는 반드시 raw_text, masked_text, masked_word 키를 포함해야 합니다.
"""

output_file = generate_text(
    dataset_name="금융 고객 상담 마스킹",
    topic="금융 고객 상담에서 개인정보를 마스킹한 텍스트 데이터셋",
    count=10,
    output_csv_path="data/masking_customer_support.csv",
    system_prompt=system_prompt
)

print(f"생성된 파일: {output_file}")
preview_df = pd.read_csv(output_file)
preview_df.head()

생성된 파일: data/masking_customer_support.csv


,raw_text,masked_text,masked_word
0,"안녕하세요, 고객님. 귀하의 계좌 번호는 123-456-7890입니다.","안녕하세요, 고객님. 귀하의 계좌 번호는 ***-***-****입니다.",123-456-7890
1,"고객님의 이름은 김철수이며, 생년월일은 1990년 1월 1일입니다.","고객님의 이름은 ***이며, 생년월일은 ****년 **월 **일입니다.","김철수, 1990년 1월 1일"
2,"고객님, 귀하의 전화번호는 010-1234-5678입니다.","고객님, 귀하의 전화번호는 ***-****-****입니다.",010-1234-5678
3,귀하의 주소는 서울특별시 강남구 테헤란로 123입니다.,귀하의 주소는 ***특별시 ***구 ***로 ***입니다.,서울특별시 강남구 테헤란로 123
4,고객님의 이메일 주소는 example@gmail.com입니다.,고객님의 이메일 주소는 ********@gmail.com입니다.,example@gmail.com
